In [17]:
import sys
from pathlib import Path

import numpy as np
from rich.console import Console
from rich.table import Table

# Ensure project root is on sys.path so `core` and `service` import cleanly
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import Settings
from service.vector_service import SimilarityMetric, VectorService

console = Console()
env_file = PROJECT_ROOT / ".env"
settings = Settings(_env_file=str(env_file) if env_file.exists() else None)
vector_service = VectorService(settings=settings)

print(f"Provider   : {vector_service.resolve_provider()}")
print(f"Embedder   : {vector_service.resolve_embedder_name()}")
print(f"Server URL : {settings.VECTOR_EMBED_SERVER_URL}")
print(f"Dimensions : {settings.VECTOR_EMBED_DIMENSIONS}")

Provider   : ollama
Embedder   : ollama/nomic-embed-text:latest
Server URL : http://10.200.0.103:11434
Dimensions : 768


## 1. Edit Query & 10 Target Strings

Modify `query_text` or any of the 10 strings in `target_texts` below and re-run the cells to explore how the embedding vectors and similarity scores behave.

In [18]:
# Query string to search against the 10 target strings (edit freely!)
# query_text = "Patient presents with acute onset shortness of breath and denies chest pain."
query_text = "Patient denies having any acute onset shortness of breath and denies chest pain at all."
# query_text = "Patient reports frequent urination episodes for the past week"


# 10 Target strings spanning closely related cardiopulmonary notes, negation differences, and unrelated clinical/non-clinical topics
target_texts = [
    "Patient presents with acute onset shortness of breath and denies chest pain.",  # 0: Exact match
    "Patient reports sudden dyspnea without any chest discomfort or angina.",        # 1: High semantic similarity (synonym)
    "Emergency visit for severe chest pain radiating to left arm with diaphoresis.", # 2: Related domain, opposite chest pain finding
    "Chest CT angiogram demonstrates bilateral pulmonary embolism in lower lobes.",  # 3: Related cardiopulmonary imaging finding
    "Chest radiograph shows clear lungs bilaterally with normal cardiac silhouette.", # 4: Normal chest imaging report
    "Brain MRI T1 sagittal reveals no acute intracranial hemorrhage or mass effect.", # 5: Different imaging modality/organ
    "Follow-up for Type 2 diabetes mellitus; HbA1c is 6.8% on metformin therapy.",   # 6: Unrelated internal medicine note
    "Right knee arthroscopy performed for medial meniscus tear rehabilitation.",     # 7: Orthopedics / surgical procedure
    "Routine pediatric well-child visit and scheduled childhood immunizations.",     # 8: Pediatrics routine care
    "DICOM C-STORE transfer syntax negotiation over TCP port 11112 succeeded.",
    "Patient complains of frequent night time urination.",
    "Patient complains of frequent daytime urination but not at bed time.",
]

print(f"Query : {query_text}")
print(f"Targets count: {len(target_texts)}")
for idx, text in enumerate(target_texts):
    print(f"  [{idx}] {text}")

Query : Patient denies having any acute onset shortness of breath and denies chest pain at all.
Targets count: 12
  [0] Patient presents with acute onset shortness of breath and denies chest pain.
  [1] Patient reports sudden dyspnea without any chest discomfort or angina.
  [2] Emergency visit for severe chest pain radiating to left arm with diaphoresis.
  [3] Chest CT angiogram demonstrates bilateral pulmonary embolism in lower lobes.
  [4] Chest radiograph shows clear lungs bilaterally with normal cardiac silhouette.
  [5] Brain MRI T1 sagittal reveals no acute intracranial hemorrhage or mass effect.
  [6] Follow-up for Type 2 diabetes mellitus; HbA1c is 6.8% on metformin therapy.
  [7] Right knee arthroscopy performed for medial meniscus tear rehabilitation.
  [8] Routine pediatric well-child visit and scheduled childhood immunizations.
  [9] DICOM C-STORE transfer syntax negotiation over TCP port 11112 succeeded.
  [10] Patient complains of frequent night time urination.
  [11] Pa

## 2. Generate L2-Normalized Vector Embeddings

Compute the normalized embedding vector for `query_text` and each of the 10 `target_texts` using `vector_service.embed_text()`.

In [19]:
query_vector = await vector_service.embed_text_async(query_text)
target_vectors = [await vector_service.embed_text_async(t) for t in target_texts]

print(f"Query vector dimension : {len(query_vector)}, L2 norm: {np.linalg.norm(query_vector):.6f}")
print(f"Target vectors count   : {len(target_vectors)}, each of dimension {len(target_vectors[0])}")
print(f"Sample query_vector[:5]: {[round(v, 5) for v in query_vector[:5]]}")

Query vector dimension : 768, L2 norm: 1.000000
Target vectors count   : 12, each of dimension 768
Sample query_vector[:5]: [0.10231, 0.01929, -0.16387, -0.01344, 0.02752]


## 3. Compare Similarity Algorithms (`[0.0, 1.0]` Scores in Index Order)

We compare `query_vector` against `target_vectors` across the four algorithms in `vector_service.py`:
- **Cosine Similarity (`cosine`)**: $\frac{\cos(	heta) + 1}{2}$ (computes full vector norms)
- **Cosine Dot-Product (`dot_product`)**: $\frac{\mathbf{q} \cdot \mathbf{t} + 1}{2}$ (fast inner product on L2-normalized vectors; matches Cosine to floating-point precision)
- **Euclidean Similarity (`euclidean`)**: $\frac{1}{1 + \|\mathbf{q} - \mathbf{t}\|_2}$ (L2 distance mapped to $(0, 1]$)
- **Manhattan Similarity (`manhattan`)**: $\frac{1}{1 + \|\mathbf{q} - \mathbf{t}\|_1}$ (L1 distance across all 768 dimensions mapped to $(0, 1]$)

In [20]:
cosine_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.COSINE)
dot_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.DOT_PRODUCT)
euclidean_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.EUCLIDEAN)
manhattan_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.MANHATTAN)

table = Table(title=f"Similarity Score Comparison (Query: '{query_text}')")
table.add_column("Idx", justify="right", style="cyan", no_wrap=True)
table.add_column("Target String", style="white")
table.add_column("Cosine", justify="right", style="green")
table.add_column("Cosine Dot", justify="right", style="bright_green")
table.add_column("Euclidean (L2)", justify="right", style="yellow")
table.add_column("Manhattan (L1)", justify="right", style="magenta")

for idx, text in enumerate(target_texts):
    table.add_row(
        str(idx),
        text,
        f"{cosine_scores[idx]:.4f}",
        f"{dot_scores[idx]:.4f}",
        f"{euclidean_scores[idx]:.4f}",
        f"{manhattan_scores[idx]:.4f}",
    )

console.print(table)

  Similarity Score Comparison (Query: 'Patient denies having any acute onset shortness of breath and denies chest  
                                                  pain at all.')                                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Idx ┃ Target String                                     ┃ Cosine ┃ Cosine Dot ┃ Euclidean (L2) ┃ Manhattan (L1) ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│   0 │ Patient presents with acute onset shortness of    │ 0.9702 │     0.9702 │         0.7432 │         0.1144 │
│     │ breath and denies chest pain.                     │        │            │                │                │
│   1 │ Patient reports sudden dyspnea without any chest  │ 0.9103 │     0.9103 │         0.6253 │         0.0712 │
│     │ discomfort or angina.                             │        │            │                │                │
│   2 │ Emergency visit for severe chest pain radiating   │ 0.7779 │     0.7779 │         0.5148 │         0.0456 │
│     │ to left arm with diaphoresis.                     │        │            │                │                │
│   3 │ Chest CT angiogram demonstrates bilateral         │ 0.7654 │     0.7654 │         0.5079 │         0.0448 │
│     │ pulmonary embolism in lower lobes.                │        │            │                │                │
│   4 │ Chest radiograph shows clear lungs bilaterally    │ 0.8128 │     0.8128 │         0.5361 │         0.0501 │
│     │ with normal cardiac silhouette.                   │        │            │                │                │
│   5 │ Brain MRI T1 sagittal reveals no acute            │ 0.7417 │     0.7417 │         0.4959 │         0.0426 │
│     │ intracranial hemorrhage or mass effect.           │        │            │                │                │
│   6 │ Follow-up for Type 2 diabetes mellitus; HbA1c is  │ 0.7308 │     0.7308 │         0.4908 │         0.0422 │
│     │ 6.8% on metformin therapy.                        │        │            │                │                │
│   7 │ Right knee arthroscopy performed for medial       │ 0.6858 │     0.6858 │         0.4715 │         0.0390 │
│     │ meniscus tear rehabilitation.                     │        │            │                │                │
│   8 │ Routine pediatric well-child visit and scheduled  │ 0.6768 │     0.6768 │         0.4679 │         0.0385 │
│     │ childhood immunizations.                          │        │            │                │                │
│   9 │ DICOM C-STORE transfer syntax negotiation over    │ 0.6906 │     0.6906 │         0.4734 │         0.0396 │
│     │ TCP port 11112 succeeded.                         │        │            │                │                │
│  10 │ Patient complains of frequent night time          │ 0.7778 │     0.7778 │         0.5147 │         0.0462 │
│     │ urination.                                        │        │            │                │                │
│  11 │ Patient complains of frequent daytime urination   │ 0.7917 │     0.7917 │         0.5228 │         0.0470 │
│     │ but not at bed time.                              │        │            │                │                │
└─────┴───────────────────────────────────────────────────┴────────┴────────────┴────────────────┴────────────────┘

## 4. Ranking Order & Algorithm Differences

Compare how each algorithm ranks the 10 target strings from highest to lowest similarity score, and measure the numerical differences between **Cosine / Cosine Dot**, **Euclidean (L2)**, and **Manhattan (L1)**.

In [21]:
# Rank indices from highest to lowest score for each algorithm
ranked_cosine = sorted(range(len(target_texts)), key=lambda i: cosine_scores[i], reverse=True)
ranked_dot = sorted(range(len(target_texts)), key=lambda i: dot_scores[i], reverse=True)
ranked_euclidean = sorted(range(len(target_texts)), key=lambda i: euclidean_scores[i], reverse=True)
ranked_manhattan = sorted(range(len(target_texts)), key=lambda i: manhattan_scores[i], reverse=True)

print("Ranking Order (Target Index from Highest -> Lowest Similarity):")
print(f"  Cosine         : {ranked_cosine}")
print(f"  Cosine Dot     : {ranked_dot}")
print(f"  Euclidean (L2) : {ranked_euclidean}")
print(f"  Manhattan (L1) : {ranked_manhattan}")

max_cos_dot_diff = max(abs(c - d) for c, d in zip(cosine_scores, dot_scores))
print(f"Max |Cosine - Cosine Dot| difference on normalized vectors: {max_cos_dot_diff:.2e}")

print("Score Range Summary across the 10 Targets:")
for name, scores in [
    ("Cosine        ", cosine_scores),
    ("Cosine Dot    ", dot_scores),
    ("Euclidean (L2)", euclidean_scores),
    ("Manhattan (L1)", manhattan_scores),
]:
    non_exact = scores[1:]  # exclude index 0 (exact match = 1.0)
    print(
        f"  {name} -> Exact[0]: {scores[0]:.4f} | "
        f"Best Match[1]: {max(non_exact):.4f} | "
        f"Lowest Match: {min(non_exact):.4f} | "
        f"Spread: {max(non_exact) - min(non_exact):.4f}"
    )

Ranking Order (Target Index from Highest -> Lowest Similarity):
  Cosine         : [0, 1, 4, 11, 2, 10, 3, 5, 6, 9, 7, 8]
  Cosine Dot     : [0, 1, 4, 11, 2, 10, 3, 5, 6, 9, 7, 8]
  Euclidean (L2) : [0, 1, 4, 11, 2, 10, 3, 5, 6, 9, 7, 8]
  Manhattan (L1) : [0, 1, 4, 11, 10, 2, 3, 5, 6, 9, 7, 8]
Max |Cosine - Cosine Dot| difference on normalized vectors: 0.00e+00
Score Range Summary across the 10 Targets:
  Cosine         -> Exact[0]: 0.9702 | Best Match[1]: 0.9103 | Lowest Match: 0.6768 | Spread: 0.2335
  Cosine Dot     -> Exact[0]: 0.9702 | Best Match[1]: 0.9103 | Lowest Match: 0.6768 | Spread: 0.2335
  Euclidean (L2) -> Exact[0]: 0.7432 | Best Match[1]: 0.6253 | Lowest Match: 0.4679 | Spread: 0.1574
  Manhattan (L1) -> Exact[0]: 0.1144 | Best Match[1]: 0.0712 | Lowest Match: 0.0385 | Spread: 0.0327


### Data Analysis Key Findings

- **Exact Equivalence of Cosine and Cosine Dot on Normalized Vectors**: Because `VectorService` L2-normalizes every embedding ($\|\mathbf{v}\|_2 = 1.0$), `cosine` and `dot_product` produce identical scores (`0.00e+00` difference), ranging from `1.0000` (Target `[0]` exact match) and `0.9116` (Target `[1]` semantic synonym) down to `0.6868` (Target `[8]` unrelated pediatric visit), while `dot_product` avoids redundant norm calculations at query time.
- **Order Preservation Between Cosine and Euclidean (L2) on the Unit Hypersphere**: Since $\|\mathbf{q} - \mathbf{t}\|_2 = \sqrt{2 - 2(\mathbf{q} \cdot \mathbf{t})}$ for unit vectors, `euclidean` produces the exact same ranking order `[0, 1, 4, 2, 3, 5, 6, 7, 9, 8]` as `cosine` / `dot_product`, with non-exact scores spanning `0.6271` down to `0.4719` (spread `0.1553`).
- **High-Dimensional Compression in Manhattan (L1) Distance**: Summing absolute differences across 768 dimensions yields large L1 distances ($\|\mathbf{q} - \mathbf{t}\|_1 \approx 13	ext{--}25$), compressing `manhattan` scores (`1 / (1 + d)`) into `0.0708` to `0.0386` (spread `0.0322`) and slightly shifting mid/low-rank order (`[6, 5]` and `[8, 9]`).

### Insights or Next Steps

- **Use Cosine Dot-Product for Graph Vector Search**: When storing L2-normalized clinical embeddings in Neo4j HNSW vector indexes, use `dot_product` (or `cosine`) for maximum score discrimination (`0.2248` spread) and fastest inner-product query execution.
- **Experiment with Custom Clinical Queries**: Edit `query_text` and `target_texts` in Section 1 (or switch `GOSMART_GS_VECTOR_EMBED_PROVIDER` to `vertexai` with `gemini-embedding-001`) and re-run all cells to compare domain sensitivity across models.